# Goal: Given the kmedoid_cluster, randomly generate a spike sequence (spike_type, spike_magnitude) for a day.

- For each cluster, a hotmap for the correlations for the spikes.
- A different hotmap for a different month.

In [9]:
import pandas as pd
import utils
import re

import torch
import torch.optim as optim
from torch import nn

import json
import os

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from collections import defaultdict

import utils

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [10]:
spike_type_mag_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike.csv', usecols=['LCLid', 'tstp', 'kmedoid_clusters','max', 'spike_type','spike_magnitude'])
spike_type_mag_df.to_csv('../../Data_processed/Spike_Details.csv', index=False)
# spike_type_mag_df = pd.read_csv('../../Data_processed/Spike_Details.csv')
spike_type_mag_df.head()

,LCLid,tstp,max,kmedoid_clusters,spike_type,spike_magnitude
0,MAC000002,2013-01-01 00:00:00,2.994,10,0,1.0
1,MAC000002,2013-01-01 00:30:00,2.994,10,0,1.0
2,MAC000002,2013-01-01 01:00:00,2.994,10,0,1.0
3,MAC000002,2013-01-01 01:30:00,2.994,10,0,1.0
4,MAC000002,2013-01-01 02:00:00,2.994,10,0,1.0


# First, determine how many spikes per day.

### Code to create the df

In [15]:
df = spike_type_mag_df.copy()
df['tstp'] = pd.to_datetime(df['tstp'])
df['date'] = df['tstp'].dt.date

# Initialize an empty DataFrame for the final results
final_results_df = pd.DataFrame()

# Iterating over each cluster
for cluster in df['kmedoid_clusters'].unique():
    cluster_df = df[df['kmedoid_clusters'] == cluster]

    # Get the unique periods (months)
    first_days = cluster_df['tstp'].dt.to_period('M').unique()

    cluster_result_df = pd.DataFrame()

    for period in first_days:
        # Since 'period' is the start of the month in datetime format, 
        # compare both year and month for filtering
        temp_df = cluster_df[(cluster_df['tstp'].dt.to_period('M') == period)]
        
        # Filter for spike types 2 and 3
        spike_df = temp_df[temp_df['spike_type'].isin([2, 3])]
        
        # Group by user and date, count the spikes
        spike_count_df = spike_df.groupby(['LCLid', 'date']).size().reset_index(name='spike_count')
        
        # Generate the full date range for each user within the month
        full_date_range_df = (
            temp_df[['LCLid', 'date']]
            .groupby('LCLid')
            .apply(lambda x: pd.DataFrame({
                'date': pd.date_range(start=x['date'].min(), end=x['date'].max()).date,  # Convert to date
                'LCLid': x['LCLid'].iloc[0]  # Ensure that 'LCLid' is retained
            }))
            .reset_index(drop=True)
        )

        # Convert 'date' columns in both dataframes to the same type
        full_date_range_df['date'] = pd.to_datetime(full_date_range_df['date']).dt.date
        spike_count_df['date'] = pd.to_datetime(spike_count_df['date']).dt.date
        
        # Merge the spike counts with the full date range (fill NaN with 0 spikes)
        full_spike_df = pd.merge(full_date_range_df, spike_count_df, on=['LCLid', 'date'], how='left')
        full_spike_df['spike_count'].fillna(0, inplace=True)

        # Group by spike count and count how often each spike count appears
        monthly_count = full_spike_df.groupby('spike_count').size().reset_index(name=f'month_{period.month}')

        # Only keep the first 6 spike counts (you can adjust this if necessary)
        monthly_count = monthly_count[monthly_count['spike_count'] <= 10]

        # Normalize by the total number of users (to calculate the relative frequency)
        total_users = monthly_count[f'month_{period.month}'].sum()
        monthly_count[f'month_{period.month}'] = monthly_count[f'month_{period.month}'] / total_users
        
        # Merge with cluster result data
        if cluster_result_df.empty:
            cluster_result_df = monthly_count
        else:
            cluster_result_df = pd.merge(cluster_result_df, monthly_count, on='spike_count', how='outer')

    # Add the cluster ID to the result
    cluster_result_df['kmedoid_clusters'] = cluster
    
    # Append to the final results
    if final_results_df.empty:
        final_results_df = cluster_result_df
    else:
        final_results_df = pd.concat([final_results_df, cluster_result_df], ignore_index=True)

# Fill any NaN values with 0
final_results_df.fillna(0, inplace=True)

# Reordering columns to place 'kmedoid_clusters' first
cols = ['kmedoid_clusters'] + [col for col in final_results_df.columns if col != 'kmedoid_clusters']
final_results_df = final_results_df[cols]

# Sort the final results by cluster and spike count
final_results_df.sort_values(['kmedoid_clusters', 'spike_count'], inplace=True)
final_results_df.reset_index(drop=True, inplace=True)


In [17]:
print(final_results_df.head(15))

    kmedoid_clusters  spike_count   month_1   month_2   month_3   month_4  \
0                  0          0.0  0.019614  0.034593  0.021718  0.025817   
1                  0          1.0  0.016077  0.026034  0.035452  0.052614   
2                  0          2.0  0.047267  0.053138  0.068029  0.093464   
3                  0          3.0  0.085209  0.086662  0.095497  0.128105   
4                  0          4.0  0.121543  0.130528  0.136698  0.154902   
5                  0          5.0  0.154341  0.169401  0.174066  0.165033   
6                  0          6.0  0.168167  0.165478  0.155541  0.148039   
7                  0          7.0  0.147588  0.140157  0.132226  0.112418   
8                  0          8.0  0.119614  0.101641  0.097413  0.066340   
9                  0          9.0  0.080386  0.061698  0.052060  0.035948   
10                 0         10.0  0.040193  0.030670  0.031300  0.017320   
11                 1          0.0  0.023490  0.030771  0.031518  0.046080   

In [8]:
final_results_df.to_csv('../../Data_processed/Spike_count_cluster_month.csv', index=False)

# Second, determine when the spike(s) should occur

In [4]:
df = spike_type_mag_df[['LCLid', 'tstp', 'kmedoid_clusters', 'spike_type']].copy()
df['tstp'] = pd.to_datetime(df['tstp'])
df['time'] = df['tstp'].dt.time
df['date'] = df['tstp'].dt.date
df = df.drop(columns=['tstp'], axis=1)
print(df.head())

       LCLid  kmedoid_clusters  spike_type      time        date
0  MAC000002                10           0  00:00:00  2013-01-01
1  MAC000002                10           0  00:30:00  2013-01-01
2  MAC000002                10           0  01:00:00  2013-01-01
3  MAC000002                10           0  01:30:00  2013-01-01
4  MAC000002                10           0  02:00:00  2013-01-01


In [5]:
daily_spike_count = df[df['spike_type'].isin([2, 3])].groupby(['LCLid', 'date']).size().reset_index(name='spike_count')
print(daily_spike_count['spike_count'].value_counts())
print(daily_spike_count['spike_count'].value_counts()[1] / daily_spike_count['spike_count'].value_counts().sum())
print(daily_spike_count['spike_count'].value_counts()[2] / daily_spike_count['spike_count'].value_counts().sum())
print(daily_spike_count['spike_count'].value_counts()[3] / daily_spike_count['spike_count'].value_counts().sum())
print(daily_spike_count['spike_count'].value_counts()[4] / daily_spike_count['spike_count'].value_counts().sum())

spike_count
1     594013
2     351594
3     136266
4      40889
5       9935
6       2216
7        435
8        116
9         19
10         9
12         6
11         5
14         3
13         1
Name: count, dtype: int64
0.5231257931479066
0.30963613610484125
0.12000454422561904
0.03600946537537857


Spike number = 1, 2, 3, 4 accounts for 95% of the users, so only use these.

In [6]:
# Function to manually parse and format the time_quad string
def parse_and_format_time_quad(time_quad_str):
    # Extract all time occurrences using regex
    times = re.findall(r'\d+, \d+', time_quad_str)
    # Convert each found time into a formatted string
    formatted_times = [f"{int(hour):02d}:{int(minute):02d}:00" for hour, minute in (time.split(', ') for time in times)]
    return tuple(formatted_times)

## When there is only one spike

In [7]:
one_spike_df = daily_spike_count[daily_spike_count['spike_count'] == 1]

df_filtered_for_one_spike = pd.merge(df, one_spike_df[['LCLid', 'date']], on=['LCLid', 'date'])

print(df_filtered_for_one_spike.head())
print(df_filtered_for_one_spike.shape)
print(df_filtered_for_one_spike['spike_type'].value_counts())

       LCLid  kmedoid_clusters  spike_type      time        date
0  MAC000002                10           0  00:00:00  2013-01-02
1  MAC000002                10           0  00:30:00  2013-01-02
2  MAC000002                10           0  01:00:00  2013-01-02
3  MAC000002                10           0  01:30:00  2013-01-02
4  MAC000002                10           0  02:00:00  2013-01-02
(28505505, 5)
spike_type
0    25274000
4     1436619
1     1200873
3      341124
2      252889
Name: count, dtype: int64


### Code for making the dataframe

In [8]:
clusters = df_filtered_for_one_spike['kmedoid_clusters'].unique()
one_spike_prob_df = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_one_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
    spikes_df = cluster_df[cluster_df['spike_type'].isin([2, 3])]
    spikes_count = spikes_df.groupby('time').size()
    total_count = cluster_df.groupby('time').size()
    probabilities = spikes_count / total_count
    prob_df = probabilities.reset_index(name='probability')
    prob_df.rename(columns={'index': 'time'}, inplace=True)
    prob_df['time'] = pd.to_datetime(prob_df['time'], format='%H:%M:%S', errors='coerce').dt.time
    # Filter out non-half-hour times
    prob_df_filtered = prob_df[prob_df['time'].apply(lambda x: x.minute == 0 or x.minute == 30)].reset_index(drop=True)
    all_times = pd.DataFrame({'time': pd.date_range("00:00", "23:30", freq="30T").time})
    final_df = pd.merge(all_times, prob_df_filtered, on='time', how='left')
    # Rename the time column to 'time_quad' and fill NaN values with 0
    final_df.rename(columns={'time': 'time_pairs'}, inplace=True)
    final_df['clusters'] = cluster
    if one_spike_prob_df.empty:
        one_spike_prob_df = final_df
    else:
        one_spike_prob_df = pd.concat([one_spike_prob_df, final_df], ignore_index=True)

print(one_spike_prob_df)
one_spike_prob_df.to_csv('../../Data_processed/statistics/1spike_time_pair.csv', index=False)


    time_pairs  probability  clusters
0     00:00:00     0.005179        10
1     00:30:00     0.003913        10
2     01:00:00     0.003333        10
3     01:30:00     0.002899        10
4     02:00:00     0.002671        10
..         ...          ...       ...
955   21:30:00     0.025631         7
956   22:00:00     0.017490         7
957   22:30:00     0.017867         7
958   23:00:00     0.016732         7
959   23:30:00     0.013491         7

[960 rows x 3 columns]


## When there are two spikes

In [9]:
two_spike_df = daily_spike_count[daily_spike_count['spike_count'] == 2]
df_filtered_for_two_spike = pd.merge(df, two_spike_df[['LCLid', 'date']], on=['LCLid', 'date'])

print(df_filtered_for_two_spike.head())
print(df_filtered_for_two_spike.shape)
print(df_filtered_for_two_spike['spike_type'].value_counts())

       LCLid  kmedoid_clusters  spike_type      time        date
0  MAC000002                10           0  00:00:00  2013-01-01
1  MAC000002                10           0  00:30:00  2013-01-01
2  MAC000002                10           0  01:00:00  2013-01-01
3  MAC000002                10           0  01:30:00  2013-01-01
4  MAC000002                10           0  02:00:00  2013-01-01
(16873337, 5)
spike_type
0    13464758
4     1465279
1     1240112
3      378428
2      324760
Name: count, dtype: int64


### Code for making the dataframe

In [10]:
two_spike_prob_df = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_two_spike[df_filtered_for_two_spike['kmedoid_clusters'] == cluster]
    spikes_df = cluster_df[cluster_df['spike_type'].isin([2, 3])]
    time_pairs_df = pd.DataFrame(columns=['LCLid', 'date', 'time_pair'])
    new_rows = []
    for (lclid, date), group in spikes_df.groupby(['LCLid', 'date']):
        time_pairs = list(combinations(group['time'], 2))  # Get all pairs of spike times
        for pair in time_pairs:
            new_rows.append({'LCLid': lclid, 'date': date, 'time_pairs': pair})

    # Convert new_rows to a DataFrame and concatenate with the existing DataFrame
    new_rows_df = pd.DataFrame(new_rows)
    time_pairs_df = pd.concat([time_pairs_df, new_rows_df], ignore_index=True)
    time_pair_counts = time_pairs_df.groupby('time_pairs').size().reset_index(name='count')
    total_time_pairs = time_pair_counts['count'].sum()
    time_pair_counts['probability'] = time_pair_counts['count'] / total_time_pairs
    time_pair_counts = time_pair_counts.drop(columns=['count'], axis=1)
    time_pair_counts['clusters'] = cluster
    if two_spike_prob_df.empty:
        two_spike_prob_df = time_pair_counts
    else:
        two_spike_prob_df = pd.concat([two_spike_prob_df, time_pair_counts], ignore_index=True)

print(two_spike_prob_df)
two_spike_prob_df.to_csv('../../Data_processed/statistics/2spike_time_pair.csv', index=False)

                 time_pairs  probability  clusters
0      (00:00:00, 01:00:00)     0.000228        10
1      (00:00:00, 01:30:00)     0.000098        10
2      (00:00:00, 02:00:00)     0.000065        10
3      (00:00:00, 03:30:00)     0.000033        10
4      (00:00:00, 04:00:00)     0.000033        10
...                     ...          ...       ...
18334  (21:30:00, 23:00:00)     0.000908         7
18335  (21:30:00, 23:30:00)     0.000681         7
18336  (22:00:00, 23:00:00)     0.001703         7
18337  (22:00:00, 23:30:00)     0.001476         7
18338  (22:30:00, 23:30:00)     0.000908         7

[18339 rows x 3 columns]


### Code for reading the dataframe

In [11]:
two_spike_prob_df = pd.read_csv('../../Data_processed/statistics/2spike_time_pair.csv')
two_spike_prob_df['time_pairs'] = two_spike_prob_df['time_pairs'].apply(parse_and_format_time_quad)
two_spike_prob_df.head()

,time_pairs,probability,clusters
0,"(00:00:00, 01:00:00)",0.000228,10
1,"(00:00:00, 01:30:00)",0.000098,10
2,"(00:00:00, 02:00:00)",0.000065,10
3,"(00:00:00, 03:30:00)",0.000033,10
4,"(00:00:00, 04:00:00)",0.000033,10


## When there are three spikes

In [12]:
three_spike_df = daily_spike_count[daily_spike_count['spike_count'] == 3]
df_filtered_for_three_spike = pd.merge(df, three_spike_df[['LCLid', 'date']], on=['LCLid', 'date'])

print(df_filtered_for_three_spike.head())
print(df_filtered_for_three_spike.shape)
print(df_filtered_for_three_spike['spike_type'].value_counts())

       LCLid  kmedoid_clusters  spike_type      time        date
0  MAC000002                10           0  00:00:00  2013-01-24
1  MAC000002                10           0  00:30:00  2013-01-24
2  MAC000002                10           0  01:00:00  2013-01-24
3  MAC000002                10           0  01:30:00  2013-01-24
4  MAC000002                10           0  02:00:00  2013-01-24
(6539607, 5)
spike_type
0    4714490
4     769518
1     646801
3     211751
2     197047
Name: count, dtype: int64


### Code for making the dataframe

In [13]:
three_spike_prob_df = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_three_spike[df_filtered_for_three_spike['kmedoid_clusters'] == cluster]
    spikes_df = cluster_df[cluster_df['spike_type'].isin([2, 3])]
    
    time_triples_df = pd.DataFrame(columns=['LCLid', 'date', 'time_pairs'])
    new_rows = []
    for (lclid, date), group in spikes_df.groupby(['LCLid', 'date']):
        time_triples = list(combinations(group['time'], 3))  # Get all triples of spike times
        for triple in time_triples:
            new_rows.append({'LCLid': lclid, 'date': date, 'time_pairs': triple})

    # Convert new_rows to a DataFrame and concatenate with the existing DataFrame
    new_rows_df = pd.DataFrame(new_rows)
    time_triples_df = pd.concat([time_triples_df, new_rows_df], ignore_index=True)
    time_triple_counts = time_triples_df.groupby('time_pairs').size().reset_index(name='count')
    total_time_triples = time_triple_counts['count'].sum()
    time_triple_counts['probability'] = time_triple_counts['count'] / total_time_triples
    time_triple_counts = time_triple_counts.drop(columns=['count'], axis=1)
    time_triple_counts['clusters'] = cluster
    if three_spike_prob_df.empty:
        three_spike_prob_df = time_triple_counts
    else:
        three_spike_prob_df = pd.concat([three_spike_prob_df, time_triple_counts], ignore_index=True)

print(three_spike_prob_df)

three_spike_prob_df.to_csv('../../Data_processed/statistics/3spike_time_pair.csv', index=False)

                           time_pairs  probability  clusters
0      (00:00:00, 01:00:00, 08:30:00)     0.000079        10
1      (00:00:00, 01:00:00, 10:30:00)     0.000079        10
2      (00:00:00, 01:00:00, 11:30:00)     0.000079        10
3      (00:00:00, 01:00:00, 16:30:00)     0.000079        10
4      (00:00:00, 01:00:00, 18:30:00)     0.000079        10
...                               ...          ...       ...
72813  (20:30:00, 21:30:00, 23:00:00)     0.000269         7
72814  (20:30:00, 22:00:00, 23:30:00)     0.000269         7
72815  (20:30:00, 22:30:00, 23:30:00)     0.000269         7
72816  (21:00:00, 22:00:00, 23:30:00)     0.000269         7
72817  (21:30:00, 22:30:00, 23:30:00)     0.000269         7

[72818 rows x 3 columns]


### Code for reading the dataframe

In [14]:
three_spike_prob_df = pd.read_csv('../../Data_processed/statistics/3spike_time_pair.csv')
three_spike_prob_df['time_pairs'] = three_spike_prob_df['time_pairs'].apply(parse_and_format_time_quad)
three_spike_prob_df.head()

,time_pairs,probability,clusters
0,"(00:00:00, 01:00:00, 08:30:00)",0.000079,10
1,"(00:00:00, 01:00:00, 10:30:00)",0.000079,10
2,"(00:00:00, 01:00:00, 11:30:00)",0.000079,10
3,"(00:00:00, 01:00:00, 16:30:00)",0.000079,10
4,"(00:00:00, 01:00:00, 18:30:00)",0.000079,10


## When there are four spikes

In [15]:
four_spike_df = daily_spike_count[daily_spike_count['spike_count'] == 4]
df_filtered_for_four_spike = pd.merge(df, four_spike_df[['LCLid', 'date']], on=['LCLid', 'date'])

print(df_filtered_for_four_spike.head())
print(df_filtered_for_four_spike.shape)
print(df_filtered_for_four_spike['spike_type'].value_counts())

       LCLid  kmedoid_clusters  spike_type      time        date
0  MAC000002                10           0  00:00:00  2013-06-07
1  MAC000002                10           0  00:30:00  2013-06-07
2  MAC000002                10           0  01:00:00  2013-06-07
3  MAC000002                10           0  01:30:00  2013-06-07
4  MAC000002                10           0  02:00:00  2013-06-07
(1962388, 5)
spike_type
0    1273906
4     286550
1     238376
3      83029
2      80527
Name: count, dtype: int64


### Code for making the dataframe

In [16]:
four_spike_prob_df = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_four_spike[df_filtered_for_four_spike['kmedoid_clusters'] == cluster]
    spikes_df = cluster_df[cluster_df['spike_type'].isin([2, 3])]
    time_quads_df = pd.DataFrame(columns=['LCLid', 'date', 'time_pairs'])
    new_rows = []
    for (lclid, date), group in spikes_df.groupby(['LCLid', 'date']):
        time_quads = list(combinations(group['time'], 4))  # Get all quads of spike times
        for quad in time_quads:
            new_rows.append({'LCLid': lclid, 'date': date, 'time_pairs': quad})

    # Convert new_rows to a DataFrame and concatenate with the existing DataFrame
    new_rows_df = pd.DataFrame(new_rows)
    time_quads_df = pd.concat([time_quads_df, new_rows_df], ignore_index=True)
    time_quad_counts = time_quads_df.groupby('time_pairs').size().reset_index(name='count')
    total_time_quads = time_quad_counts['count'].sum()
    time_quad_counts['probability'] = time_quad_counts['count'] / total_time_quads
    time_quad_counts = time_quad_counts.drop(columns=['count'], axis=1)
    time_quad_counts['clusters'] = cluster
    if four_spike_prob_df.empty:
        four_spike_prob_df = time_quad_counts
    else:
        four_spike_prob_df = pd.concat([four_spike_prob_df, time_quad_counts], ignore_index=True)

print(four_spike_prob_df)
four_spike_prob_df.to_csv('../../Data_processed/statistics/4spike_time_pair.csv', index=False)

                                     time_pairs  probability  clusters
0      (00:00:00, 01:00:00, 09:00:00, 13:00:00)     0.000255        10
1      (00:00:00, 01:00:00, 10:00:00, 20:30:00)     0.000255        10
2      (00:00:00, 01:00:00, 11:00:00, 19:00:00)     0.000255        10
3      (00:00:00, 01:00:00, 12:30:00, 19:30:00)     0.000255        10
4      (00:00:00, 01:00:00, 13:00:00, 16:00:00)     0.000255        10
...                                         ...          ...       ...
39057  (17:00:00, 18:30:00, 19:30:00, 23:00:00)     0.000818         7
39058  (17:00:00, 19:00:00, 21:30:00, 22:30:00)     0.000818         7
39059  (18:00:00, 19:30:00, 22:00:00, 23:00:00)     0.000818         7
39060  (18:30:00, 19:30:00, 20:30:00, 23:00:00)     0.000818         7
39061  (19:30:00, 21:00:00, 22:00:00, 23:30:00)     0.000818         7

[39062 rows x 3 columns]


### Code for reading the dataframe

In [17]:
four_spike_prob_df = pd.read_csv('../../Data_processed/statistics/4spike_time_pair.csv')
four_spike_prob_df['time_pairs'] = four_spike_prob_df['time_pairs'].apply(parse_and_format_time_quad)
four_spike_prob_df.head()

,time_pairs,probability,clusters
0,"(00:00:00, 01:00:00, 09:00:00, 13:00:00)",0.000255,10
1,"(00:00:00, 01:00:00, 10:00:00, 20:30:00)",0.000255,10
2,"(00:00:00, 01:00:00, 11:00:00, 19:00:00)",0.000255,10
3,"(00:00:00, 01:00:00, 12:30:00, 19:30:00)",0.000255,10
4,"(00:00:00, 01:00:00, 13:00:00, 16:00:00)",0.000255,10


# Third, determine the spike type: first-order or second-order, and determine the number of pre-spikes/post-spikes before/after the two types of spikes.

## For one spike per day

In [18]:
df_filtered_for_one_spike['spike_type'].value_counts()

spike_type
0    25274000
4     1436619
1     1200873
3      341124
2      252889
Name: count, dtype: int64

count numbers
1       234
2       123

In [19]:
spike_type_prob_df_1 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_one_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
    spike_2_count = cluster_df['spike_type'].value_counts()[2]
    spike_3_count = cluster_df['spike_type'].value_counts()[3]
    total_count = spike_2_count + spike_3_count
    spike_2_prob = spike_2_count / total_count
    spike_3_prob = spike_3_count / total_count
    cluster_prob_df = pd.DataFrame({'spike_type': [2, 3], 'probability': [spike_2_prob, spike_3_prob]})
    cluster_prob_df['clusters'] = cluster
    cluster_prob_df = cluster_prob_df[['clusters', 'spike_type', 'probability']]
    if spike_type_prob_df_1.empty:
        spike_type_prob_df_1 = cluster_prob_df
    else:
        spike_type_prob_df_1 = pd.concat([spike_type_prob_df_1, cluster_prob_df], ignore_index=True)

print(spike_type_prob_df_1)
spike_type_prob_df_1.to_csv('../../Data_processed/statistics/1spike_type_prob.csv', index=False)

    clusters  spike_type  probability
0         10           2     0.486026
1         10           3     0.513974
2          6           2     0.534120
3          6           3     0.465880
4          4           2     0.338769
5          4           3     0.661231
6         11           2     0.519221
7         11           3     0.480779
8         18           2     0.354033
9         18           3     0.645967
10        17           2     0.544620
11        17           3     0.455380
12         1           2     0.478426
13         1           3     0.521574
14        16           2     0.440136
15        16           3     0.559864
16        15           2     0.317550
17        15           3     0.682450
18         2           2     0.424967
19         2           3     0.575033
20        19           2     0.375818
21        19           3     0.624182
22         3           2     0.354022
23         3           3     0.645978
24         5           2     0.404781
25         5

In [20]:
# Adjusted logic to count the number of consecutive 1s before a 2
count_ones_before_two_1 = defaultdict(int)
count_ones_before_three_1 = defaultdict(int)
count_fours_after_two_1 = defaultdict(int)
count_fours_after_three_1 = defaultdict(int)

cluster_df_12_1 = pd.DataFrame()
cluster_df_13_1 = pd.DataFrame()
cluster_df_24_1 = pd.DataFrame()
cluster_df_34_1 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_one_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
    for index, row in cluster_df.iterrows():
        if row['spike_type'] == 2:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_two_1[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_two_1[count_4s] += 1
        elif row['spike_type'] == 3:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_three_1[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_three_1[count_4s] += 1
    count_ones_before_two_1_df = pd.DataFrame(count_ones_before_two_1.items(), columns=['count_1s_before_2', 'frequency'])
    count_ones_before_two_1_df.sort_values('count_1s_before_2', inplace=True)
    count_ones_before_two_1_df['probability'] = count_ones_before_two_1_df['frequency'] / count_ones_before_two_1_df['frequency'].sum()
    count_ones_before_two_1_df = count_ones_before_two_1_df.drop(columns=['frequency'], axis=1)
    count_ones_before_two_1_df['cluster'] = cluster
    # Only keep the first 7 counts because the rest are negligible
    count_ones_before_two_1_df = count_ones_before_two_1_df.head(7)
    count_ones_before_two_1_df['probability'] /= count_ones_before_two_1_df['probability'].sum()

    count_ones_before_three_1_df = pd.DataFrame(count_ones_before_three_1.items(), columns=['count_1s_before_3', 'frequency'])
    count_ones_before_three_1_df.sort_values('count_1s_before_3', inplace=True)
    count_ones_before_three_1_df['probability'] = count_ones_before_three_1_df['frequency'] / count_ones_before_three_1_df['frequency'].sum()
    count_ones_before_three_1_df = count_ones_before_three_1_df.drop(columns=['frequency'], axis=1)
    count_ones_before_three_1_df['cluster'] = cluster
    count_ones_before_three_1_df = count_ones_before_three_1_df.head(7)
    count_ones_before_three_1_df['probability'] /= count_ones_before_three_1_df['probability'].sum()

    count_fours_after_two_1_df = pd.DataFrame(count_fours_after_two_1.items(), columns=['count_4s_after_2', 'frequency'])
    count_fours_after_two_1_df.sort_values('count_4s_after_2', inplace=True)
    count_fours_after_two_1_df['probability'] = count_fours_after_two_1_df['frequency'] / count_fours_after_two_1_df['frequency'].sum()
    count_fours_after_two_1_df = count_fours_after_two_1_df.drop(columns=['frequency'], axis=1)
    count_fours_after_two_1_df['cluster'] = cluster
    count_fours_after_two_1_df = count_fours_after_two_1_df.head(7)
    count_fours_after_two_1_df['probability'] /= count_fours_after_two_1_df['probability'].sum()

    count_fours_after_three_1_df = pd.DataFrame(count_fours_after_three_1.items(), columns=['count_4s_after_3', 'frequency'])
    count_fours_after_three_1_df.sort_values('count_4s_after_3', inplace=True)
    count_fours_after_three_1_df['probability'] = count_fours_after_three_1_df['frequency'] / count_fours_after_three_1_df['frequency'].sum()
    count_fours_after_three_1_df = count_fours_after_three_1_df.drop(columns=['frequency'], axis=1)
    count_fours_after_three_1_df['cluster'] = cluster
    count_fours_after_three_1_df = count_fours_after_three_1_df.head(7)
    count_fours_after_three_1_df['probability'] /= count_fours_after_three_1_df['probability'].sum()

    if cluster_df_12_1.empty:
        cluster_df_12_1 = count_ones_before_two_1_df
        cluster_df_13_1 = count_ones_before_three_1_df
        cluster_df_24_1 = count_fours_after_two_1_df
        cluster_df_34_1 = count_fours_after_three_1_df
    else:
        cluster_df_12_1 = pd.concat([cluster_df_12_1, count_ones_before_two_1_df], ignore_index=True)
        cluster_df_13_1 = pd.concat([cluster_df_13_1, count_ones_before_three_1_df], ignore_index=True)
        cluster_df_24_1 = pd.concat([cluster_df_24_1, count_fours_after_two_1_df], ignore_index=True)
        cluster_df_34_1 = pd.concat([cluster_df_34_1, count_fours_after_three_1_df], ignore_index=True)

cluster_df_12_1.to_csv('../../Data_processed/statistics/1spike_12_clusters.csv', index=False)
cluster_df_13_1.to_csv('../../Data_processed/statistics/1spike_13_clusters.csv', index=False)
cluster_df_24_1.to_csv('../../Data_processed/statistics/1spike_24_clusters.csv', index=False)
cluster_df_34_1.to_csv('../../Data_processed/statistics/1spike_34_clusters.csv', index=False)
print(cluster_df_12_1)
# cluster_df_12_1 = pd.read_csv('../../Data_processed/statistics/1spike_12_clusters.csv')
# cluster_df_13_1 = pd.read_csv('../../Data_processed/statistics/1spike_13_clusters.csv')
# cluster_df_24_1 = pd.read_csv('../../Data_processed/statistics/1spike_24_clusters.csv')
# cluster_df_34_1 = pd.read_csv('../../Data_processed/statistics/1spike_34_clusters.csv')

     count_1s_before_2  probability  cluster
0                    1     0.584546       10
1                    2     0.212766       10
2                    3     0.083987       10
3                    4     0.050392       10
4                    5     0.035834       10
..                 ...          ...      ...
135                  3     0.087438        7
136                  4     0.049562        7
137                  5     0.034149        7
138                  6     0.022061        7
139                  7     0.014707        7

[140 rows x 3 columns]


## For two spikes per day

In [21]:
df_filtered_for_two_spike['spike_type'].value_counts()

spike_type
0    13464758
4     1465279
1     1240112
3      378428
2      324760
Name: count, dtype: int64

In [22]:
spike_type_prob_df_2 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_two_spike[df_filtered_for_two_spike['kmedoid_clusters'] == cluster]
    spike_2_count = cluster_df['spike_type'].value_counts()[2]
    spike_3_count = cluster_df['spike_type'].value_counts()[3]
    total_count = spike_2_count + spike_3_count
    spike_2_prob = spike_2_count / total_count
    spike_3_prob = spike_3_count / total_count
    cluster_prob_df = pd.DataFrame({'spike_type': [2, 3], 'probability': [spike_2_prob, spike_3_prob]})
    cluster_prob_df['clusters'] = cluster
    cluster_prob_df = cluster_prob_df[['clusters', 'spike_type', 'probability']]
    if spike_type_prob_df_2.empty:
        spike_type_prob_df_2 = cluster_prob_df
    else:
        spike_type_prob_df_2 = pd.concat([spike_type_prob_df_2, cluster_prob_df], ignore_index=True)

print(spike_type_prob_df_2)
spike_type_prob_df_2.to_csv('../../Data_processed/statistics/2spike_type_prob.csv', index=False)
spike_type_prob_df_2 = pd.read_csv('../../Data_processed/statistics/2spike_type_prob.csv')

    clusters  spike_type  probability
0         10           2     0.495217
1         10           3     0.504783
2          6           2     0.568245
3          6           3     0.431755
4          4           2     0.374304
5          4           3     0.625696
6         11           2     0.541404
7         11           3     0.458596
8         18           2     0.402084
9         18           3     0.597916
10        17           2     0.551111
11        17           3     0.448889
12         1           2     0.501416
13         1           3     0.498584
14        16           2     0.497417
15        16           3     0.502583
16        15           2     0.367849
17        15           3     0.632151
18         2           2     0.446867
19         2           3     0.553133
20        19           2     0.416969
21        19           3     0.583031
22         3           2     0.402099
23         3           3     0.597901
24         5           2     0.442204
25         5

In [23]:
# Adjusted logic to count the number of consecutive 1s before a 2
count_ones_before_two_2 = defaultdict(int)
count_ones_before_three_2 = defaultdict(int)
count_fours_after_two_2 = defaultdict(int)
count_fours_after_three_2 = defaultdict(int)

cluster_df_12_2 = pd.DataFrame()
cluster_df_13_2 = pd.DataFrame()
cluster_df_24_2 = pd.DataFrame()
cluster_df_34_2 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_two_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
    for index, row in cluster_df.iterrows():
        if row['spike_type'] == 2:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_two_2[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_two_2[count_4s] += 1
        elif row['spike_type'] == 3:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_three_2[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_three_2[count_4s] += 1
    count_ones_before_two_2_df = pd.DataFrame(count_ones_before_two_2.items(), columns=['count_1s_before_2', 'frequency'])
    count_ones_before_two_2_df.sort_values('count_1s_before_2', inplace=True)
    count_ones_before_two_2_df['probability'] = count_ones_before_two_2_df['frequency'] / count_ones_before_two_2_df['frequency'].sum()
    count_ones_before_two_2_df = count_ones_before_two_2_df.drop(columns=['frequency'], axis=1)
    count_ones_before_two_2_df['cluster'] = cluster
    count_ones_before_two_2_df = count_ones_before_two_2_df.head(7)
    count_ones_before_two_2_df['probability'] /= count_ones_before_two_2_df['probability'].sum()

    count_ones_before_three_2_df = pd.DataFrame(count_ones_before_three_2.items(), columns=['count_1s_before_3', 'frequency'])
    count_ones_before_three_2_df.sort_values('count_1s_before_3', inplace=True)
    count_ones_before_three_2_df['probability'] = count_ones_before_three_2_df['frequency'] / count_ones_before_three_2_df['frequency'].sum()
    count_ones_before_three_2_df = count_ones_before_three_2_df.drop(columns=['frequency'], axis=1)
    count_ones_before_three_2_df['cluster'] = cluster
    count_ones_before_three_2_df = count_ones_before_three_2_df.head(7)
    count_ones_before_three_2_df['probability'] /= count_ones_before_three_2_df['probability'].sum()

    count_fours_after_two_2_df = pd.DataFrame(count_fours_after_two_2.items(), columns=['count_4s_after_2', 'frequency'])
    count_fours_after_two_2_df.sort_values('count_4s_after_2', inplace=True)
    count_fours_after_two_2_df['probability'] = count_fours_after_two_2_df['frequency'] / count_fours_after_two_2_df['frequency'].sum()
    count_fours_after_two_2_df = count_fours_after_two_2_df.drop(columns=['frequency'], axis=1)
    count_fours_after_two_2_df['cluster'] = cluster
    count_fours_after_two_2_df = count_fours_after_two_2_df.head(7)
    count_fours_after_two_2_df['probability'] /= count_fours_after_two_2_df['probability'].sum()

    count_fours_after_three_2_df = pd.DataFrame(count_fours_after_three_2.items(), columns=['count_4s_after_3', 'frequency'])
    count_fours_after_three_2_df.sort_values('count_4s_after_3', inplace=True)
    count_fours_after_three_2_df['probability'] = count_fours_after_three_2_df['frequency'] / count_fours_after_three_2_df['frequency'].sum()
    count_fours_after_three_2_df = count_fours_after_three_2_df.drop(columns=['frequency'], axis=1)
    count_fours_after_three_2_df['cluster'] = cluster
    count_fours_after_three_2_df = count_fours_after_three_2_df.head(7)
    count_fours_after_three_2_df['probability'] /= count_fours_after_three_2_df['probability'].sum()

    if cluster_df_12_2.empty:
        cluster_df_12_2 = count_ones_before_two_2_df
        cluster_df_13_2 = count_ones_before_three_2_df
        cluster_df_24_2 = count_fours_after_two_2_df
        cluster_df_34_2 = count_fours_after_three_2_df
    else:
        cluster_df_12_2 = pd.concat([cluster_df_12_2, count_ones_before_two_2_df], ignore_index=True)
        cluster_df_13_2 = pd.concat([cluster_df_13_2, count_ones_before_three_2_df], ignore_index=True)
        cluster_df_24_2 = pd.concat([cluster_df_24_2, count_fours_after_two_2_df], ignore_index=True)
        cluster_df_34_2 = pd.concat([cluster_df_34_2, count_fours_after_three_2_df], ignore_index=True)

cluster_df_12_2.to_csv('../../Data_processed/statistics/2spike_12_clusters.csv', index=False)
cluster_df_13_2.to_csv('../../Data_processed/statistics/2spike_13_clusters.csv', index=False)
cluster_df_24_2.to_csv('../../Data_processed/statistics/2spike_24_clusters.csv', index=False)
cluster_df_34_2.to_csv('../../Data_processed/statistics/2spike_34_clusters.csv', index=False)
print(cluster_df_12_2)
# cluster_df_12_2 = pd.read_csv('../../Data_processed/statistics/2spike_12_clusters.csv')
# cluster_df_13_2 = pd.read_csv('../../Data_processed/statistics/2spike_13_clusters.csv')
# cluster_df_24_2 = pd.read_csv('../../Data_processed/statistics/2spike_24_clusters.csv')
# cluster_df_34_2 = pd.read_csv('../../Data_processed/statistics/2spike_34_clusters.csv')

C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1788195151.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_two_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1788195151.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_two_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1788195151.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_two_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1788195151.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_two_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local

     count_1s_before_2  probability  cluster
0                    1     0.578404       10
1                    2     0.220657       10
2                    3     0.079812       10
3                    4     0.044131       10
4                    5     0.028169       10
..                 ...          ...      ...
135                  3     0.092064        7
136                  4     0.050580        7
137                  5     0.029878        7
138                  6     0.020546        7
139                  7     0.014664        7

[140 rows x 3 columns]


## For three spikes per day

In [24]:
df_filtered_for_three_spike['spike_type'].value_counts()

spike_type
0    4714490
4     769518
1     646801
3     211751
2     197047
Name: count, dtype: int64

In [25]:
spike_type_prob_df_3 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_three_spike[df_filtered_for_three_spike['kmedoid_clusters'] == cluster]
    spike_2_count = cluster_df['spike_type'].value_counts()[2]
    spike_3_count = cluster_df['spike_type'].value_counts()[3]
    total_count = spike_2_count + spike_3_count
    spike_2_prob = spike_2_count / total_count
    spike_3_prob = spike_3_count / total_count
    cluster_prob_df = pd.DataFrame({'spike_type': [2, 3], 'probability': [spike_2_prob, spike_3_prob]})
    cluster_prob_df['clusters'] = cluster
    cluster_prob_df = cluster_prob_df[['clusters', 'spike_type', 'probability']]
    if spike_type_prob_df_3.empty:
        spike_type_prob_df_3 = cluster_prob_df
    else:
        spike_type_prob_df_3 = pd.concat([spike_type_prob_df_3, cluster_prob_df], ignore_index=True)

print(spike_type_prob_df_3)
spike_type_prob_df_3.to_csv('../../Data_processed/statistics/3spike_type_prob.csv', index=False)
spike_type_prob_df_3 = pd.read_csv('../../Data_processed/statistics/3spike_type_prob.csv')

    clusters  spike_type  probability
0         10           2     0.500673
1         10           3     0.499327
2          6           2     0.575863
3          6           3     0.424137
4          4           2     0.378899
5          4           3     0.621101
6         11           2     0.549398
7         11           3     0.450602
8         18           2     0.427769
9         18           3     0.572231
10        17           2     0.555568
11        17           3     0.444432
12         1           2     0.513202
13         1           3     0.486798
14        16           2     0.544551
15        16           3     0.455449
16        15           2     0.386113
17        15           3     0.613887
18         2           2     0.451436
19         2           3     0.548564
20        19           2     0.438128
21        19           3     0.561872
22         3           2     0.422425
23         3           3     0.577575
24         5           2     0.472880
25         5

In [26]:
count_ones_before_two_3 = defaultdict(int)
count_ones_before_three_3 = defaultdict(int)
count_fours_after_two_3 = defaultdict(int)
count_fours_after_three_3 = defaultdict(int)

cluster_df_12_3 = pd.DataFrame()
cluster_df_13_3 = pd.DataFrame()
cluster_df_24_3 = pd.DataFrame()
cluster_df_34_3 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_three_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
    for index, row in cluster_df.iterrows():
        if row['spike_type'] == 2:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_two_3[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_two_3[count_4s] += 1
        elif row['spike_type'] == 3:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_three_3[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_three_3[count_4s] += 1
    count_ones_before_two_3_df = pd.DataFrame(count_ones_before_two_3.items(), columns=['count_1s_before_2', 'frequency'])
    count_ones_before_two_3_df.sort_values('count_1s_before_2', inplace=True)
    count_ones_before_two_3_df['probability'] = count_ones_before_two_3_df['frequency'] / count_ones_before_two_3_df['frequency'].sum()
    count_ones_before_two_3_df = count_ones_before_two_3_df.drop(columns=['frequency'], axis=1)
    count_ones_before_two_3_df['cluster'] = cluster
    count_ones_before_two_3_df = count_ones_before_two_3_df.head(7)
    count_ones_before_two_3_df['probability'] /= count_ones_before_two_3_df['probability'].sum()

    count_ones_before_three_3_df = pd.DataFrame(count_ones_before_three_3.items(), columns=['count_1s_before_3', 'frequency'])
    count_ones_before_three_3_df.sort_values('count_1s_before_3', inplace=True)
    count_ones_before_three_3_df['probability'] = count_ones_before_three_3_df['frequency'] / count_ones_before_three_3_df['frequency'].sum()
    count_ones_before_three_3_df = count_ones_before_three_3_df.drop(columns=['frequency'], axis=1)
    count_ones_before_three_3_df['cluster'] = cluster
    count_ones_before_three_3_df = count_ones_before_three_3_df.head(7)
    count_ones_before_three_3_df['probability'] /= count_ones_before_three_3_df['probability'].sum()

    count_fours_after_two_3_df = pd.DataFrame(count_fours_after_two_3.items(), columns=['count_4s_after_2', 'frequency'])
    count_fours_after_two_3_df.sort_values('count_4s_after_2', inplace=True)
    count_fours_after_two_3_df['probability'] = count_fours_after_two_3_df['frequency'] / count_fours_after_two_3_df['frequency'].sum()
    count_fours_after_two_3_df = count_fours_after_two_3_df.drop(columns=['frequency'], axis=1)
    count_fours_after_two_3_df['cluster'] = cluster
    count_fours_after_two_3_df = count_fours_after_two_3_df.head(7)
    count_fours_after_two_3_df['probability'] /= count_fours_after_two_3_df['probability'].sum()

    count_fours_after_three_3_df = pd.DataFrame(count_fours_after_three_3.items(), columns=['count_4s_after_3', 'frequency'])
    count_fours_after_three_3_df.sort_values('count_4s_after_3', inplace=True)
    count_fours_after_three_3_df['probability'] = count_fours_after_three_3_df['frequency'] / count_fours_after_three_3_df['frequency'].sum()
    count_fours_after_three_3_df = count_fours_after_three_3_df.drop(columns=['frequency'], axis=1)
    count_fours_after_three_3_df['cluster'] = cluster
    count_fours_after_three_3_df = count_fours_after_three_3_df.head(7)
    count_fours_after_three_3_df['probability'] /= count_fours_after_three_3_df['probability'].sum()

    if cluster_df_12_3.empty:
        cluster_df_12_3 = count_ones_before_two_3_df
        cluster_df_13_3 = count_ones_before_three_3_df
        cluster_df_24_3 = count_fours_after_two_3_df
        cluster_df_34_3 = count_fours_after_three_3_df
    else:
        cluster_df_12_3 = pd.concat([cluster_df_12_3, count_ones_before_two_3_df], ignore_index=True)
        cluster_df_13_3 = pd.concat([cluster_df_13_3, count_ones_before_three_3_df], ignore_index=True)
        cluster_df_24_3 = pd.concat([cluster_df_24_3, count_fours_after_two_3_df], ignore_index=True)
        cluster_df_34_3 = pd.concat([cluster_df_34_3, count_fours_after_three_3_df], ignore_index=True)


cluster_df_12_3.to_csv('../../Data_processed/statistics/3spike_12_clusters.csv', index=False)
cluster_df_13_3.to_csv('../../Data_processed/statistics/3spike_13_clusters.csv', index=False)
cluster_df_24_3.to_csv('../../Data_processed/statistics/3spike_24_clusters.csv', index=False)
cluster_df_34_3.to_csv('../../Data_processed/statistics/3spike_34_clusters.csv', index=False)

print(cluster_df_12_3)

C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1917892904.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_three_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1917892904.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_three_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1917892904.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_three_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1917892904.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_three_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppDa

     count_1s_before_2  probability  cluster
0                    1     0.577107       10
1                    2     0.214626       10
2                    3     0.108108       10
3                    4     0.034976       10
4                    5     0.033386       10
..                 ...          ...      ...
135                  3     0.090422        7
136                  4     0.048765        7
137                  5     0.031803        7
138                  6     0.019332        7
139                  7     0.011349        7

[140 rows x 3 columns]


# For four spikes per day

In [27]:
df_filtered_for_four_spike['spike_type'].value_counts()

spike_type
0    1273906
4     286550
1     238376
3      83029
2      80527
Name: count, dtype: int64

In [28]:
spike_type_prob_df_4 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_four_spike[df_filtered_for_four_spike['kmedoid_clusters'] == cluster]
    spike_2_count = cluster_df['spike_type'].value_counts()[2]
    spike_3_count = cluster_df['spike_type'].value_counts()[3]
    total_count = spike_2_count + spike_3_count
    spike_2_prob = spike_2_count / total_count
    spike_3_prob = spike_3_count / total_count
    cluster_prob_df = pd.DataFrame({'spike_type': [2, 3], 'probability': [spike_2_prob, spike_3_prob]})
    cluster_prob_df['clusters'] = cluster
    cluster_prob_df = cluster_prob_df[['clusters', 'spike_type', 'probability']]
    if spike_type_prob_df_4.empty:
        spike_type_prob_df_4 = cluster_prob_df
    else:
        spike_type_prob_df_4 = pd.concat([spike_type_prob_df_4, cluster_prob_df], ignore_index=True)

print(spike_type_prob_df_4)
spike_type_prob_df_4.to_csv('../../Data_processed/statistics/4spike_type_prob.csv', index=False)
spike_type_prob_df_4 = pd.read_csv('../../Data_processed/statistics/4spike_type_prob.csv')

    clusters  spike_type  probability
0         10           2     0.504787
1         10           3     0.495213
2          6           2     0.601271
3          6           3     0.398729
4          4           2     0.366308
5          4           3     0.633692
6         11           2     0.563112
7         11           3     0.436888
8         18           2     0.446423
9         18           3     0.553577
10        17           2     0.567666
11        17           3     0.432334
12         1           2     0.504919
13         1           3     0.495081
14        16           2     0.574760
15        16           3     0.425240
16        15           2     0.405146
17        15           3     0.594854
18         2           2     0.459347
19         2           3     0.540653
20        19           2     0.445019
21        19           3     0.554981
22         3           2     0.416573
23         3           3     0.583427
24         5           2     0.479053
25         5

In [29]:
count_ones_before_two_4 = defaultdict(int)
count_ones_before_three_4 = defaultdict(int)
count_fours_after_two_4 = defaultdict(int)
count_fours_after_three_4 = defaultdict(int)

cluster_df_12_4 = pd.DataFrame()
cluster_df_13_4 = pd.DataFrame()
cluster_df_24_4 = pd.DataFrame()
cluster_df_34_4 = pd.DataFrame()

for cluster in clusters:
    cluster_df = df_filtered_for_four_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
    for index, row in cluster_df.iterrows():
        if row['spike_type'] == 2:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_two_4[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_two_4[count_4s] += 1
        elif row['spike_type'] == 3:
            count_1s = 0
            i = index - 1
            while i >= 0 and df.iloc[i]['spike_type'] == 1:
                count_1s += 1
                i -= 1
            if count_1s > 0:
                count_ones_before_three_4[count_1s] += 1
            count_4s = 0
            j = index + 1
            while j < cluster_df.shape[0] and df.iloc[j]['spike_type'] == 4:
                count_4s += 1
                j += 1
            if count_4s > 0:
                count_fours_after_three_4[count_4s] += 1
    count_ones_before_two_4_df = pd.DataFrame(count_ones_before_two_4.items(), columns=['count_1s_before_2', 'frequency'])
    count_ones_before_two_4_df.sort_values('count_1s_before_2', inplace=True)
    count_ones_before_two_4_df['probability'] = count_ones_before_two_4_df['frequency'] / count_ones_before_two_4_df['frequency'].sum()
    count_ones_before_two_4_df = count_ones_before_two_4_df.drop(columns=['frequency'], axis=1)
    count_ones_before_two_4_df['cluster'] = cluster
    count_ones_before_two_4_df = count_ones_before_two_4_df.head(7)
    count_ones_before_two_4_df['probability'] /= count_ones_before_two_4_df['probability'].sum()

    count_ones_before_three_4_df = pd.DataFrame(count_ones_before_three_4.items(), columns=['count_1s_before_3', 'frequency'])
    count_ones_before_three_4_df.sort_values('count_1s_before_3', inplace=True)
    count_ones_before_three_4_df['probability'] = count_ones_before_three_4_df['frequency'] / count_ones_before_three_4_df['frequency'].sum()
    count_ones_before_three_4_df = count_ones_before_three_4_df.drop(columns=['frequency'], axis=1)
    count_ones_before_three_4_df['cluster'] = cluster
    count_ones_before_three_4_df = count_ones_before_three_4_df.head(7)
    count_ones_before_three_4_df['probability'] /= count_ones_before_three_4_df['probability'].sum()

    count_fours_after_two_4_df = pd.DataFrame(count_fours_after_two_4.items(), columns=['count_4s_after_2', 'frequency'])
    count_fours_after_two_4_df.sort_values('count_4s_after_2', inplace=True)
    count_fours_after_two_4_df['probability'] = count_fours_after_two_4_df['frequency'] / count_fours_after_two_4_df['frequency'].sum()
    count_fours_after_two_4_df = count_fours_after_two_4_df.drop(columns=['frequency'], axis=1)
    count_fours_after_two_4_df['cluster'] = cluster
    count_fours_after_two_4_df = count_fours_after_two_4_df.head(7)
    count_fours_after_two_4_df['probability'] /= count_fours_after_two_4_df['probability'].sum()

    count_fours_after_three_4_df = pd.DataFrame(count_fours_after_three_4.items(), columns=['count_4s_after_3', 'frequency'])
    count_fours_after_three_4_df.sort_values('count_4s_after_3', inplace=True)
    count_fours_after_three_4_df['probability'] = count_fours_after_three_4_df['frequency'] / count_fours_after_three_4_df['frequency'].sum()
    count_fours_after_three_4_df = count_fours_after_three_4_df.drop(columns=['frequency'], axis=1)
    count_fours_after_three_4_df['cluster'] = cluster
    count_fours_after_three_4_df = count_fours_after_three_4_df.head(7)
    count_fours_after_three_4_df['probability'] /= count_fours_after_three_4_df['probability'].sum()

    if cluster_df_12_4.empty:
        cluster_df_12_4 = count_ones_before_two_4_df
        cluster_df_13_4 = count_ones_before_three_4_df
        cluster_df_24_4 = count_fours_after_two_4_df
        cluster_df_34_4 = count_fours_after_three_4_df
    else:
        cluster_df_12_4 = pd.concat([cluster_df_12_4, count_ones_before_two_4_df], ignore_index=True)
        cluster_df_13_4 = pd.concat([cluster_df_13_4, count_ones_before_three_4_df], ignore_index=True)
        cluster_df_24_4 = pd.concat([cluster_df_24_4, count_fours_after_two_4_df], ignore_index=True)
        cluster_df_34_4 = pd.concat([cluster_df_34_4, count_fours_after_three_4_df], ignore_index=True)


cluster_df_12_4.to_csv('../../Data_processed/statistics/4spike_12_clusters.csv', index=False)
cluster_df_13_4.to_csv('../../Data_processed/statistics/4spike_13_clusters.csv', index=False)
cluster_df_24_4.to_csv('../../Data_processed/statistics/4spike_24_clusters.csv', index=False)
cluster_df_34_4.to_csv('../../Data_processed/statistics/4spike_34_clusters.csv', index=False)
print(cluster_df_12_4)
# cluster_df_12_4 = pd.read_csv('../../Data_processed/statistics/4spike_12_clusters.csv')
# cluster_df_13_4 = pd.read_csv('../../Data_processed/statistics/4spike_13_clusters.csv')
# cluster_df_24_4 = pd.read_csv('../../Data_processed/statistics/4spike_24_clusters.csv')
# cluster_df_34_4 = pd.read_csv('../../Data_processed/statistics/4spike_34_clusters.csv')

C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1953065606.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_four_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1953065606.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_four_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1953065606.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_four_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\Local\Temp\ipykernel_33456\1953065606.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cluster_df = df_filtered_for_four_spike[df_filtered_for_one_spike['kmedoid_clusters'] == cluster]
C:\Users\edwar\AppData\L

     count_1s_before_2  probability  cluster
0                    1     0.528239       10
1                    2     0.239203       10
2                    3     0.093023       10
3                    4     0.066445       10
4                    5     0.043189       10
..                 ...          ...      ...
135                  3     0.098335        7
136                  4     0.051480        7
137                  5     0.029901        7
138                  6     0.022811        7
139                  7     0.014180        7

[140 rows x 3 columns]


# Lastly, determine the magnitude of the spikes

In [30]:
spike_type_mag_df.head()

,LCLid,tstp,max,kmedoid_clusters,spike_type,spike_magnitude
0,MAC000002,2013-01-01 00:00:00,1.65425,10,0,1.0
1,MAC000002,2013-01-01 00:30:00,1.65425,10,0,1.0
2,MAC000002,2013-01-01 01:00:00,1.65425,10,0,1.0
3,MAC000002,2013-01-01 01:30:00,1.65425,10,0,1.0
4,MAC000002,2013-01-01 02:00:00,1.65425,10,0,1.0


In [31]:
mag_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike.csv')
mag_df.head()

,LCLid,tstp,energy(kWh/hh)_smoothed,energy(kWh/hh),mean,median,std,min,max,gradient,kmedoid_clusters,is_medoid,spike_type,spike_magnitude,temperature,windSpeed,humidity
0,MAC000002,2013-01-01 00:00:00,0.226083,0.219,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,10,0,0,1.0,0.33250,0.3672,0.6494
1,MAC000002,2013-01-01 00:30:00,0.222500,0.241,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,10,0,0,1.0,0.33885,0.3689,0.6429
2,MAC000002,2013-01-01 01:00:00,0.220000,0.191,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,10,0,0,1.0,0.34520,0.3706,0.6364
3,MAC000002,2013-01-01 01:30:00,0.208250,0.235,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,10,0,0,1.0,0.34085,0.3784,0.6104
4,MAC000002,2013-01-01 02:00:00,0.198167,0.182,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,10,0,0,1.0,0.33650,0.3862,0.5844


In [32]:
clusters = mag_df['kmedoid_clusters'].unique()

spike_mag_stats_df = pd.DataFrame()

for cluster in clusters:
    cluster_df = mag_df[mag_df['kmedoid_clusters'] == cluster]
    spike_2_data = cluster_df[cluster_df["spike_type"] == 2]["spike_magnitude"]
    spike_3_data = cluster_df[cluster_df["spike_type"] == 3]["spike_magnitude"]
    spike_2_mean = spike_2_data.mean()
    spike_3_mean = spike_3_data.mean()
    spike_2_std = spike_2_data.std()
    spike_3_std = spike_3_data.std()

    spike_mag_stats = pd.DataFrame({'spike_type': [2, 3], 'mean': [spike_2_mean, spike_3_mean], 'std': [spike_2_std, spike_3_std]})
    spike_mag_stats['clusters'] = cluster

    if spike_mag_stats_df.empty:
        spike_mag_stats_df = spike_mag_stats
    else:
        spike_mag_stats_df = pd.concat([spike_mag_stats_df, spike_mag_stats], ignore_index=True)

spike_mag_stats_df = spike_mag_stats_df[['clusters', 'spike_type', 'mean', 'std']]

print(spike_mag_stats_df)
spike_mag_stats_df.to_csv('../../Data_processed/statistics/spike_mag_stats.csv', index=False)


    clusters  spike_type      mean       std
0         10           2  1.114704  0.266786
1         10           3  1.645400  0.534882
2          6           2  1.984470  0.762771
3          6           3  2.870793  1.096522
4          4           2  0.441355  0.115194
5          4           3  0.821838  0.338650
6         11           2  1.194200  0.218434
7         11           3  1.691400  0.424817
8         18           2  0.508409  0.114160
9         18           3  0.857776  0.297192
10        17           2  1.425449  0.377550
11        17           3  2.057266  0.679914
12         1           2  0.931356  0.147489
13         1           3  1.390477  0.385395
14        16           2  0.276701  0.081529
15        16           3  0.521189  0.249901
16        15           2  0.494703  0.109069
17        15           3  0.893037  0.322441
18         2           2  0.835515  0.204617
19         2           3  1.320674  0.469489
20        19           2  0.420124  0.102438
21        

In [36]:
def random_spike_mag(cluster, spike_type, mean, std, max):
    '''
    Function to generate random spike magnitude based on the mean and standard deviation of the cluster
    '''

    if spike_type == 2:
        min_mag = mean + 2 * std
        max_mag = mean + 3 * std
    else:
        min_mag = mean + 3 * std
        max_mag = max
    spike_mag_stats_df = pd.read_csv('../../Data_processed/statistics/spike_mag_stats.csv')
    cluster_df = spike_mag_stats_df[spike_mag_stats_df['clusters'] == cluster]
    cluster_df = cluster_df[cluster_df['spike_type'] == spike_type]
    mean = cluster_df['mean'].values[0]
    std = cluster_df['std'].values[0]
    spike_mag = np.random.normal(mean, std)
    while spike_mag < min_mag or spike_mag > max_mag:
        spike_mag = np.random.normal(mean, std)
    return spike_mag



In [37]:
random_spike_mag(0, 2, 0, 1, 10)

KeyboardInterrupt: 